# LC 208 — Implement Trie (Prefix Tree)

<div style="border-left:4px solid purple; padding:10px 16px;
background:#f5f0ff; margin-top:12px">
<strong>Core Insight:</strong> A Trie stores strings
character-by-character in a tree where each node represents
one character. Shared prefixes share nodes, so prefix lookups
are O(m) where m is the word length — never O(n·m).
</div>

## Official Problem Statement

A **trie** (pronounced as "try") or **prefix tree** is a tree
data structure used to efficiently store and retrieve keys in a
dataset of strings.

Implement the `Trie` class:

- `Trie()` — initializes the trie object.
- `void insert(String word)` — inserts the string `word` into
  the trie.
- `boolean search(String word)` — returns `true` if the string
  `word` is in the trie (i.e., was inserted before), and
  `false` otherwise.
- `boolean startsWith(String prefix)` — returns `true` if
  there is a previously inserted string that has the prefix
  `prefix`, and `false` otherwise.

**Constraints:**
- `1 <= word.length, prefix.length <= 2000`
- `word` and `prefix` consist only of lowercase English
  letters.
- At most `3 * 10^4` calls in total to `insert`, `search`,
  and `startsWith`.

## What This Is Actually Asking

Build a data structure that stores a set of words and can
answer two kinds of queries: exact-word membership and
prefix-existence.

A hash set could answer exact membership in O(1), but it
cannot answer prefix queries without scanning every stored
word.

The Trie solves both in O(m) time by walking one character
at a time down a tree whose edges are labeled by characters.

The only difference between `search` and `startsWith` is
that `search` additionally checks the `is_end` flag at the
final node; `startsWith` just confirms the path exists.

## Walk Through an Example by Hand

Operations: insert("apple"), search("apple"),
search("app"), startsWith("app"), insert("app"),
search("app")

**insert("apple")**
```
root -> 'a' -> 'p' -> 'p' -> 'l' -> 'e'[END]
```
Walk root. 'a' missing → create node. 'p' missing → create.
'p' missing → create. 'l' missing → create.
'e' missing → create, set is_end=True.

**search("apple")** → walk a→p→p→l→e, node exists,
is_end=True → **True**

**search("app")** → walk a→p→p, node exists,
is_end=False → **False**

**startsWith("app")** → walk a→p→p, node exists
(don't check is_end) → **True**

**insert("app")** → walk a→p→p (all exist), set
is_end=True on 'p' node.

**search("app")** → walk a→p→p, is_end=True → **True**

## The Picture

After inserting: "apple", "app", "apply", "bat"

```
           root
          /    \
        'a'    'b'
         |      |
        'p'    'a'
         |      |
        'p'*   't'*
        / \
      'l' (end of "app")
      / \
    'e'  'y'
    [*]  [*]

  * = is_end True
  [*] = leaf that is also end
```

Key observations:
- "app" and "apple"/"apply" **share** the a→p→p path.
- The `is_end` flag marks where a complete word finishes.
- `startsWith("ap")` → True (path a→p exists).
- `search("ap")` → False (is_end is False at 'p' node).
- `startsWith("ba")` → True; `startsWith("bc")` → False.

## When To Use This Pattern

- When you need **prefix-based lookups** (autocomplete,
  spell-check), think **Trie**.
- When multiple strings share common prefixes and memory
  compression matters, think **Trie**.
- When you must answer "does any stored word start with X?"
  in O(|X|), think **Trie**.
- When building a router or DNS resolver that matches
  hierarchical string paths, think **Trie**.
- When a hash set is too slow for prefix queries or too
  memory-hungry for many shared-prefix strings, think
  **Trie**.

## The Approach

Each node holds a dictionary `children` mapping a character
to its child node, plus a boolean `is_end` flag.

For `insert`: starting at root, iterate over each character.
If the character is not in `children`, create a new node.
Move current pointer to that child. After the last character,
set `is_end = True`.

For `search`: same walk, but if any character is missing
return False immediately. After the last character, return
`current.is_end`.

For `startsWith`: identical to `search` walk, but return
True as long as the path exists — ignore `is_end`.

In [ ]:
# Standard type hints
from typing import Optional, Dict

In [ ]:
def test_harness(cls):
    """
    Replay (op, args, expected) sequences.
    'None' expected values are skipped (constructor).
    """
    sequences = [
        # (op, args, expected)
        [
            ("Trie",       [],          None),
            ("insert",     ["apple"],   None),
            ("search",     ["apple"],   True),
            ("search",     ["app"],     False),
            ("startsWith", ["app"],     True),
            ("insert",     ["app"],     None),
            ("search",     ["app"],     True),
        ],
        [
            ("Trie",       [],          None),
            ("insert",     ["bat"],     None),
            ("insert",     ["ball"],    None),
            ("search",     ["ball"],    True),
            ("search",     ["bal"],     False),
            ("startsWith", ["ba"],      True),
            ("startsWith", ["bc"],      False),
            ("search",     ["bat"],     True),
        ],
        [
            ("Trie",       [],          None),
            ("search",     ["missing"], False),
            ("startsWith", ["mi"],      False),
        ],
    ]

    passed = 0
    failed = 0

    for s_idx, seq in enumerate(sequences):
        obj = None
        print(f"--- Sequence {s_idx + 1} ---")
        for op, args, expected in seq:
            if op == "Trie":
                obj = cls()
                result = None
            else:
                result = getattr(obj, op)(*args)

            if expected is None:
                print(f"  {op}({args}) -> (void)")
                continue

            ok = result == expected
            status = "PASSED" if ok else "FAILED"
            if ok:
                passed += 1
            else:
                failed += 1
            print(
                f"  {status} | {op}({args})"
                f" expected={expected} got={result}"
            )

    total = passed + failed
    print(f"\nResult: {passed}/{total} passed, "
          f"{failed}/{total} failed.")

In [ ]:
class TrieNode:
    """
    A single node in the Trie.

    Attributes
    ----------
    children : dict[str, TrieNode]
        Maps a character to its child TrieNode.
    is_end : bool
        True if this node marks the end of a complete
        inserted word.
    """
    def __init__(self):
        self.children: Dict[str, "TrieNode"] = {}
        self.is_end: bool = False


class Trie:
    """
    Prefix tree supporting insert, exact-search,
    and prefix-search in O(m) time per operation
    where m is the length of the word/prefix.

    Methods
    -------
    insert(word: str) -> None
        Add word to the trie.
    search(word: str) -> bool
        Return True if word was inserted.
    startsWith(prefix: str) -> bool
        Return True if any inserted word starts
        with prefix.
    """

    def __init__(self):
        """
        Initialize trie with an empty root node.
        """
        print("[DEBUG] Trie initialized with empty root.")
        self.root = TrieNode()

    def insert(self, word: str) -> None:
        """
        Insert word into the trie.

        Walk from root, creating child nodes as
        needed for each character. Mark is_end on
        the final node.

        Parameters
        ----------
        word : str
            The word to insert.
        """
        print(f"[DEBUG] insert('{word}')")
        pass

    def search(self, word: str) -> bool:
        """
        Return True if word exists in the trie.

        Walk from root character by character. If
        any character is missing, return False.
        After the last character, return is_end.

        Parameters
        ----------
        word : str
            The word to search for.

        Returns
        -------
        bool
        """
        print(f"[DEBUG] search('{word}')")
        pass

    def startsWith(self, prefix: str) -> bool:
        """
        Return True if any inserted word starts with
        prefix.

        Same walk as search, but return True as soon
        as the full prefix path is confirmed — do not
        inspect is_end.

        Parameters
        ----------
        prefix : str
            The prefix to test.

        Returns
        -------
        bool
        """
        print(f"[DEBUG] startsWith('{prefix}')")
        pass

In [ ]:
# Uncomment and run when solution is ready
# test_harness(Trie)

## Complexity

Let **m** = length of word/prefix, **n** = number of words
stored, **k** = average word length.

| Approach | insert | search | startsWith | Space |
|----------|--------|--------|------------|-------|
| Hash set | O(m) | O(m) | O(n·m) scan | O(n·k) |
| **Trie (optimal)** | **O(m)** | **O(m)** | **O(m)** | **O(n·k)** |

- Time per operation is O(m) regardless of how many words
  are stored.
- Space is O(n·k) in the worst case (no shared prefixes);
  better when many words share prefixes.
- Each node holds at most 26 children for lowercase English
  (can use array instead of dict for speed).

## Real World Connection

At **Citi**, trade surveillance systems must rapidly check
whether a ticker symbol or CUSIP matches any entry in a
watchlist; a Trie enables O(m) lookups over millions of
instruments without scanning the full list.

**AWS CloudFront** and API Gateway use prefix trees to
match URL path prefixes against routing rules — the same
Trie principle routes `/api/v2/users` to the correct
Lambda in microseconds.

In **data engineering**, schema registries and Glue
catalog lookups for column-name autocomplete in Athena
queries benefit from Trie-backed prefix search,
especially when column namespaces share long common
prefixes like `customer_profile_v2_`.

DNS resolvers also walk a Trie of domain name labels
(in reverse: `com` → `example` → `www`) to resolve
hostnames to IP addresses in O(depth) time.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra